# Notebook 06 — Evaluation & Comparison
Load all result JSON files, build comparison table, generate charts for the report.

In [ ]:
import sys, json
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from src.evaluation.visualize import load_results, plot_comparison_bar


## Load All Results

In [ ]:
results = load_results('../results/scores')
print('Methods found:', list(results.keys()))

# Rename keys for display
display_results = {
    'TextRank':  results.get('TEXTRANK', {}),
    'BART':      results.get('BART', {}),
    'PEGASUS':   results.get('PEGASUS', {}),
    'Flan-T5':   results.get('FLANT5', {}),
}
display_results = {k: v for k, v in display_results.items() if v}


## Comparison Table

In [ ]:
metrics = ['rouge1', 'rouge2', 'rougeL', 'bertscore_f1', 'bleu']
table = pd.DataFrame(display_results).T[metrics].rename(columns={
    'rouge1': 'ROUGE-1', 'rouge2': 'ROUGE-2', 'rougeL': 'ROUGE-L',
    'bertscore_f1': 'BERTScore F1', 'bleu': 'BLEU-4'
})
print(table.to_string())
print()
# LaTeX table for the paper
print(table.to_latex(float_format='%.4f', bold_rows=True))


## Bar Chart — ROUGE Metrics

In [ ]:
fig = plot_comparison_bar(
    display_results,
    metrics=['rouge1', 'rouge2', 'rougeL'],
    save_path='../results/figures/rouge_comparison.png',
    title='ROUGE Score Comparison — DialogSum Test Set',
)
plt.show()


## Bar Chart — BERTScore & BLEU

In [ ]:
fig = plot_comparison_bar(
    display_results,
    metrics=['bertscore_f1', 'bleu'],
    save_path='../results/figures/bertscore_bleu_comparison.png',
    title='BERTScore F1 & BLEU-4 Comparison',
)
plt.show()


## Qualitative Examples — Side-by-side

In [ ]:
# Load predictions from each method (stored as lists in results JSON if saved)
# If only scores were saved, re-run inference here or load from a predictions JSON
from src.data.loader import load_dialogsum
dataset = load_dialogsum(cache_dir='../data/raw/dialogsum')
test = dataset['test']
examples = [0, 5, 100]

for idx in examples:
    print(f'=== Sample {idx} ===')
    print('Dialogue :', test['dialogue'][idx][:300])
    print('Reference:', test['summary'][idx])
    print()


## Summary
Best method should be BART fine-tuned on ROUGE-L and BERTScore-F1, demonstrating that domain-adapted abstractive models outperform both extractive methods and zero-shot instruction tuning.